In [ ]:
#Tabela Sidra
cultivos_dict = {"Soja (em grão)": "2713","Milho (em grão)": "2711","Cana-de-açúcar": "2696","Algodão herbáceo (em caroço)": "2689","Mandioca": "2708","Arroz (em casca)": "2692","Feijão (em grão)": "2702"}
2689,2693,2696,2702,2708,2711,2713
anos = [str(a) for a in range(2004, 2026)]
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ""AppleWebKit/537.36"}
def limpiar_texto(texto):
    if isinstance(texto, str):
        return (unidecode(texto).lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_"))
    return texto
#Funcao de descarga: cultura + un año
def descargar_ano_cultivo(args):
    ano, nome_cultura_limpo, codigo_api = args
    url = (f"https://apisidra.ibge.gov.br/values/t/1612/"f"n6/all/v/allxp/p/{ano}/c81/{codigo_api}")
    try:
        response = requests.get(url,headers=headers,timeout=60)
        if response.status_code == 200:
            df_raw = pd.read_json(io.StringIO(response.text))
            df_clean = df_raw.iloc[1:].copy() #eliminação cabeçalho
            df_organizada = df_clean.pivot(index=["D1C", "D1N", "D3N"],columns="D2N",values="V").reset_index()
            df_organizada.columns.name = None
            df_organizada = df_organizada.rename(columns={"D1C": "code_muni","D1N": "municipio","D3N": "ano"})
            df_organizada.columns = [limpiar_texto(col)for col in df_organizada.columns]
            df_organizada["cultura"] = nome_cultura_limpo
            return df_organizada
        else:
            print(f"HTTP {response.status_code} | "f"{nome_cultura_limpo} | {ano}")
    except Exception as e:
        print(f"Erro na descarga "f"{nome_cultura_limpo} - Ano {ano}: {e}")
    return None
lista_dataframes_maestros = []
for cultura_original, codigo in cultivos_dict.items():
    cultura_limpia = limpiar_texto(cultura_original)
    print(f"\n---> Descarregado: "f"{cultura_limpia} | Codigo: {codigo}")

    tareas = [
        (ano, cultura_limpia, codigo)
        for ano in anos
    ]
    with ThreadPoolExecutor(max_workers=4) as executor:
        resultados = executor.map(descargar_ano_cultivo,tareas)
        for res in resultados:
            if res is not None:
                lista_dataframes_maestros.append(res)
df = pd.concat(lista_dataframes_maestros,ignore_index=True)
df["code_muni"] = (df["code_muni"].astype(str))
df["ano"] = pd.to_numeric(df["ano"],errors="coerce")
indices = ["code_muni", "municipio", "ano"]
indicadores = ["quantidade_produzida","rendimento_medio_da_producao","valor_da_producao","area_colhida","area_plantada"]
df = df.pivot(index=indices,columns="cultura",values=indicadores)
df.columns = [f"{cultura}_{indicador}"for indicador, cultura in df.columns]
df = df.reset_index()
df = df.sort_values(by=["code_muni", "ano"]).reset_index(drop=True)
df